# Practice: NFXP under the hood

ECO 629, Class 11 (Tuesday, September 29). Lecture notes: <https://dse.iskh.me/practical-nfxp>

Four exercises on the NFXP estimator of Class 10, taken from Exercise set 3 of the
*Dynamic Programming* course at the University of Copenhagen (Spring 2020) by Patrick
Kofod Mogensen and Maria Juul Hansen. The questions are quoted from it; the code you
write uses `zurcher.py` and `nfxp.py` in this folder.

## Update the environment

The profiler exercise needs `line_profiler`, which is now in `requirements.txt` and
`environment.yml` at the root of the code repository. Run the cell below once: it
updates the environment this notebook runs in, with conda if the kernel is a conda
environment and with uv otherwise. Restart the kernel afterwards.

In [ ]:
import os, sys, subprocess
from pathlib import Path

root = Path.cwd().parent                                  # the root of sb-dse-code
if (Path(sys.prefix) / 'conda-meta').exists():            # the kernel runs in a conda environment
    cmd = ['conda', 'env', 'update', '--prefix', sys.prefix, '--file', str(root / 'environment.yml')]
else:                                                     # a uv (or plain venv) environment
    cmd = ['uv', 'pip', 'install', '--python', sys.executable, '-r', str(root / 'requirements.txt')]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

## Setup

Load the bus data with `read_busdata()`, print what the estimator will see with
`describe_data()`, and estimate the model once with `estim_zurcher(data).estimate()`
to have the baseline in front of you.

## Exercise 1: run time profiler

Profile one call of `estim_zurcher(data).estimate()` with `cProfile` (or `%prun`),
sorted by cumulative time, and then line by line with `%load_ext line_profiler` and
`%lprun -f est.score est.estimate()`.

The optimizer options are the `method` argument of `estimate()`: `'bhhh'` and
`'trust-ncg'` use the analytical score and the BHHH approximation of the Hessian,
`'bfgs'` the analytical score and no Hessian, `'bfgs-numerical'` finite differences
instead of the score, and `'nelder-mead'` function values only. Compare the estimates,
the log-likelihood, the run time, the number of solves and the `converged` flag.

> Try using line_profiler in python. This gives you a lot of information about the
> performance of your code.
>
> - Now try changing the optimizer options, and turn the use of the non-numerical
>   Hessian off. What happens?
> - Now also try it with the analytical gradient off, what happens?

## Exercise 2: the effect of the warm start

`solve()` starts the fixed point iterations from the previous solution `self.ev`; the
switch is the `warm_start` argument of `estim_zurcher`. Compare `time`,
`n_solver_calls`, `n_inner_iter` and the estimates in the `Result` with and without it.

> We use the latest EV guess to start the nfxp.solve-procedure even though we change
> θ from one likelihood iteration to another. Why do you think we do that?
>
> - What if we started with EV = 0 in each iteration? Try that and see what happens
>   with the parameters and the numerical performance.

## Exercise 3: estimation under different β

β is an argument of `estim_zurcher`, not an element of `theta`. Estimate the model for
β in (0, 0.5, 0.9, 0.99, 0.999, 0.9999) and compare $RC$, $c$ and the log-likelihood.

> Try estimate the model for different values of β.
>
> - Why can we not estimate β?
> - When estimating with different β, do the changes in the estimates of c and/or RC
>   make intuitively sense?
> - Can you think of some data/variation, which could allow us to identify β?

## Exercise 4: alternative specification of the mileage process

`read_busdata(n=..., max_miles=...)` sets the number of bins and where the grid ends.
Estimate the model for (n, max_miles) = (175, 450 000), (350, 900 000),
(175, 900 000) and (88, 225 000), and for each print the width of a bin, the number
of transition probabilities `J`, $RC$, $c$, the log-likelihood and the share of
bus-months in the top bin. Predict each row before you run it.

> Try setting the maximum number of miles (odometer reading) to 900. Now the
> absorbing state is much higher.
>
> - If we adjust the number of grid points as well, so that we have a comparable model
>   (multiply the number of grids by 2), do we get a better fit?
> - Try to lower the number of grid points to 175 again. How do the parameters change?
>   Are the changes intuitive?
> - What if you change the max to 225 and half the number of grids (hint: what goes
>   wrong?)?